# sample_segments

> Sample segment display with first/last segments and jump-to-index

In [ ]:
#| default_exp components.sample_segments

In [ ]:
#| export
from typing import Any, List, Optional

from fasthtml.common import Div, Span, H3, Input, Button, Form

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.card import card, card_body
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input, text_input_sizes
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, font_family
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, gap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Lucide icons via factory
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_transcript_verify.html_ids import VerifyHtmlIds
from cjm_transcript_verify.models import VerificationResult, SegmentSample, VerifyUrls
from cjm_transcript_verify.utils import format_time_range

## Sample Row Component

In [ ]:
#| export
def render_sample_row(
    sample:SegmentSample,  # Segment sample to display
) -> Any:  # Sample row element
    """Render a single segment sample row."""
    timing = format_time_range(sample.start_time, sample.end_time)
    
    return Div(
        # Index badge
        Span(
            f"[{sample.index}]",
            cls=combine_classes(font_family.mono, font_size.sm, text_dui.primary)
        ),
        # Text (truncated)
        Span(
            f'"{sample.text}"',
            cls=combine_classes(font_size.sm, "truncate", "flex-1")
        ),
        # Timing
        Span(
            f"({timing})",
            cls=combine_classes(font_size.xs, text_dui.base_content.opacity(60), font_family.mono)
        ),
        cls=combine_classes(flex_display, items.center, gap(2), p.y(1))
    )

## Sample Lists

In [ ]:
#| export
def render_sample_list(
    samples:List[SegmentSample],  # Samples to display
    label:str,  # Label for this list (e.g., "First", "Last")
    container_id:str="",  # Optional container ID
) -> Any:  # Sample list element
    """Render a list of segment samples with label."""
    if not samples:
        return Div(
            Span(f"{label}:", cls=combine_classes(font_weight.medium, font_size.sm)),
            Span("No samples", cls=combine_classes(text_dui.base_content.opacity(60), font_size.sm)),
            id=container_id if container_id else None,
            cls=combine_classes(flex_display, flex_direction.col, gap(1))
        )
    
    return Div(
        Span(f"{label}:", cls=combine_classes(font_weight.medium, font_size.sm, m.b(1))),
        *[render_sample_row(s) for s in samples],
        id=container_id if container_id else None,
        cls=combine_classes(flex_display, flex_direction.col)
    )

## Jump-to-Index Input

Input field for jumping to a specific segment index. The actual route
will be wired in Phase 4.

In [ ]:
#| export
def render_jump_to_index(
    urls:VerifyUrls=None,  # URL bundle for routes (Phase 4)
    max_index:int=0,  # Maximum valid index for placeholder
) -> Any:  # Jump-to-index form
    """Render the jump-to-index input form."""
    urls = urls or VerifyUrls()
    
    # Build form attributes
    form_attrs = {}
    if urls.sample:
        form_attrs.update({
            "hx_post": urls.sample,
            "hx_target": f"#{VerifyHtmlIds.JUMP_RESULT}",
            "hx_swap": "innerHTML",
        })
    
    placeholder = f"0-{max_index}" if max_index > 0 else "index"
    
    return Div(
        Span("Jump to index:", cls=combine_classes(font_size.sm, font_weight.medium)),
        Form(
            Input(
                type="number",
                name="index",
                min="0",
                max=str(max_index) if max_index > 0 else None,
                placeholder=placeholder,
                id=VerifyHtmlIds.JUMP_INPUT,
                cls=combine_classes(text_input, text_input_sizes.sm, w(20), font_family.mono)
            ),
            Button(
                lucide_icon("arrow-right", size=4),
                type="submit",
                id=VerifyHtmlIds.JUMP_BUTTON,
                cls=combine_classes(btn, btn_colors.primary, btn_sizes.sm)
            ),
            method="post",
            cls=combine_classes(flex_display, items.center, gap(2)),
            **form_attrs
        ),
        cls=combine_classes(flex_display, items.center, gap(3), m.t(3), p.t(3), "border-t", "border-base-300")
    )

## Jump Result Container

In [ ]:
#| export
def render_jump_result(
    sample:Optional[SegmentSample]=None,  # Fetched sample or None
    error:str="",  # Error message if any
) -> Any:  # Jump result display
    """Render the jump-to-index result (single segment or error)."""
    if error:
        return Div(
            lucide_icon("circle-alert", size=4, cls=str(text_dui.error)),
            Span(error, cls=combine_classes(font_size.sm, text_dui.error)),
            id=VerifyHtmlIds.JUMP_RESULT,
            cls=combine_classes(flex_display, items.center, gap(2), m.t(2))
        )
    
    if sample:
        return Div(
            render_sample_row(sample),
            id=VerifyHtmlIds.JUMP_RESULT,
            cls=str(m.t(2))
        )
    
    # Empty placeholder
    return Div(id=VerifyHtmlIds.JUMP_RESULT)

## Full Samples Section

In [ ]:
#| export
def render_sample_segments(
    result:VerificationResult,  # Verification result with samples
    urls:VerifyUrls=None,  # URL bundle for routes
) -> Any:  # Sample segments card
    """Render the full sample segments section."""
    urls = urls or VerifyUrls()
    max_index = result.segment_count - 1 if result.segment_count > 0 else 0
    
    header = Div(
        lucide_icon("scan-search", size=5),
        H3("Sample Segments", cls=combine_classes(font_size.lg, font_weight.semibold)),
        cls=combine_classes(flex_display, items.center, gap(2), m.b(3))
    )
    
    return Div(
        Div(
            header,
            
            # First samples
            render_sample_list(
                result.first_segments,
                "First",
                VerifyHtmlIds.FIRST_SAMPLES
            ),
            
            # Spacer
            Div(cls=str(m.y(3))),
            
            # Last samples
            render_sample_list(
                result.last_segments,
                "Last",
                VerifyHtmlIds.LAST_SAMPLES
            ),
            
            # Jump-to-index
            render_jump_to_index(urls, max_index),
            
            # Result container (empty initially)
            render_jump_result(),
            
            cls=str(card_body)
        ),
        id=VerifyHtmlIds.SAMPLES_SECTION,
        cls=combine_classes(card, bg_dui.base_100, "shadow-sm")
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()